# SWaT preprocessing check

This notebook runs `prepare_swat` from `src/data_loaders/preprocessing.py` and shows what it produces. It saves nothing.

Steps in the module: drop the 6 sensors that never change in either file, min-max scale every sensor using the normal file only, then cut 60-second windows that start every 10 seconds.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_loaders.preprocessing import prepare_swat

d = prepare_swat(ROOT / "data" / "raw" / "swat")
W, S = d["window"], d["stride"]
feats = d["features"]
print("features:", len(feats), "| dropped:", d["dropped"])
print("normal windows:", d["normal_windows"].shape)
print("attack windows:", d["attack_windows"].shape)
print("attack seconds:", int(d["attack_labels"].sum()), "of", len(d["attack_labels"]))

## One window

A window is 60 consecutive seconds of all 45 sensors, one row per second. Values are scaled so the normal file runs from 0 to 1.

In [ ]:
pd.DataFrame(d["normal_windows"][0], columns=feats).head(10)

## Three sensors over the attack file

One point per window (its last second, so one point every 10 s). Red points are seconds labelled attack.
- LIT301 and FIT501 leave the normal-file range (0 to 1) only during attacks.
- P102 is a spare pump that never moves in the normal file. It is kept because it switches on during some attacks.

Three sensors (AIT201, AIT402, AIT502) drift slowly across the attack days, so they sit outside the normal-file range even in seconds labelled Normal. Their scaled values are not plotted here.

The attack file labels attack #27 (P-302 and LIT-401, start 31 Dec 01:17) as attack until 11:15, about 10 hours. The attack list gives it about 28 minutes. This one block is 35,900 of the 54,621 attack seconds (66%), so per-second scores mostly reflect it. The labels are used as released; why the label runs longer is not established. Because of this, results should also report per-attack detection (eTaPR), not only point-wise F1.

In [ ]:
n = len(d["attack_windows"])
t = d["attack_timestamps"][W - 1::S][:n]
lab = d["attack_labels"][W - 1::S][:n]
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, name in zip(axes, ["LIT301", "FIT501", "P102"]):
    y = d["attack_windows"][:, -1, feats.index(name)]
    ax.plot(t, y, lw=0.6)
    ax.scatter(t[lab == 1], y[lab == 1], s=2, c="red")
    ax.set_ylabel(name)
axes[0].set_title("Scaled sensors over the attack file (0 to 1 = normal-file range)")
plt.tight_layout()
plt.show()